# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q Faker
    print('Running in Colab')

# Outlier Detection In-Depth

Identifying and handling outliers is crucial in social science research. We explore three methods:
1.  **Z-Score**: Parametric method (assumes normality).
2.  **IQR Method**: Non-parametric method (robust to extreme values).
3.  **Isolation Forest**: Machine Learning method for high-dimensional or complex data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from faker import Faker

fake = Faker()
np.random.seed(42)

# Generate Income Data (Log-normal distribution which is naturally skewed)
n = 500
income = np.random.lognormal(mean=10.5, sigma=0.5, size=n)

# Add explicit outliers (High Net Worth Individuals)
outliers = np.random.uniform(150000, 300000, 10)
data = np.concatenate([income, outliers])

df = pd.DataFrame({'ID': range(len(data)), 'Income': data})

plt.figure(figsize=(10, 4))
sns.boxplot(x=df['Income'])
plt.title('Income Distribution with Outliers')
plt.show()

## 1. Z-Score Method
Points with a Z-score > 3 or < -3 are considered outliers. Note: This method is sensitive to the outliers themselves as they inflate the mean and SD.

In [ ]:
df['Z_Score'] = np.abs(stats.zscore(df['Income']))
outliers_z = df[df['Z_Score'] > 3]

print(f"Z-Score detected {len(outliers_z)} outliers.")
print(outliers_z[['ID', 'Income', 'Z_Score']].head())

## 2. IQR Method (Tukey's Fences)
Uses quartiles. Outliers are defined as values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

In [ ]:
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = df[(df['Income'] < lower_bound) | (df['Income'] > upper_bound)]

print(f"IQR Method detected {len(outliers_iqr)} outliers.")
print(f"Bounds: {lower_bound:.2f} to {upper_bound:.2f}")

## 3. Isolation Forest (Machine Learning)
Unsupervised learning algorithm that isolates observations by randomly selecting a feature and then randomly selecting a split value between the maximum and minimum values of the selected feature.

In [ ]:
from sklearn.ensemble import IsolationForest

# Isolation Forest expects 2D array
iso = IsolationForest(contamination=0.05, random_state=42)
df['Anomaly'] = iso.fit_predict(df[['Income']])

# Anomaly = -1 indicates outlier
outliers_iso = df[df['Anomaly'] == -1]

print(f"Isolation Forest detected {len(outliers_iso)} outliers.")

## Visualization Comparison
Highlighting outliers detected by the Isolation Forest.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='ID', y='Income', data=df, hue='Anomaly', palette={1: 'blue', -1: 'red'})
plt.title('Income Outliers (Isolation Forest)')
plt.xlabel('Participant ID')
plt.ylabel('Income')
plt.show()